In [2]:
import os
import sys

# Go 3 levels up from components → project root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.append(project_root)

print("Added to path:", project_root)

Added to path: c:\Users\Gunjan\AI_Project


In [3]:
import os
import sys
import pickle
from dataclasses import dataclass
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from src.IDS.exception import CustomException
from src.IDS.logger import logging
from dataclasses import dataclass

In [4]:
import src
print("SRC is now recognized successfully")

SRC is now recognized successfully


In [ ]:
from src.IDS.utils import read_sql_data

monday_df, tue_fri_df = read_sql_data()

print("Monday shape:", monday_df.shape)
print("Tue-Fri shape:", tue_fri_df.shape)

c:\Users\Gunjan\AI_Project\src\IDS\utils.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tue_fri_df = pd.read_sql_query("SELECT * FROM tuetofri_ids", mydb)


In [ ]:
print(tue_fri_df["Attack"].value_counts())

Attack
0    1291600
1     329485
Name: count, dtype: int64


In [ ]:
monday_df = monday_df.dropna()

if "Attack" in monday_df.columns:
    monday_df = monday_df.drop("Attack", axis=1)


In [ ]:
scaler = StandardScaler()
X_monday_scaled = scaler.fit_transform(monday_df)



In [ ]:
iso_model = IsolationForest(
    n_estimators=300,
    contamination=0.01,
    random_state=42
)

In [ ]:
iso_model.fit(X_monday_scaled)

print("Isolation Forest trained successfully")

Isolation Forest trained successfully


In [ ]:
tue_fri_df = tue_fri_df.dropna()

In [ ]:
y = tue_fri_df["Attack"].astype(int)
X = tue_fri_df.drop("Attack", axis=1)

In [ ]:
X_scaled = scaler.transform(X)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1296868, 43)
Test shape: (324217, 43)


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42
)

In [ ]:
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9988680420829259
F1 Score: 0.9972141886609128

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    258320
           1       1.00      1.00      1.00     65897

    accuracy                           1.00    324217
   macro avg       1.00      1.00      1.00    324217
weighted avg       1.00      1.00      1.00    324217


Confusion Matrix:

[[258164    156]
 [   211  65686]]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
import time

In [ ]:
start = time.time()

log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic F1:", f1_score(y_test, y_pred_log))
print("Time:", time.time() - start)

Logistic F1: 0.9166090047958769
Time: 33.74327802658081


In [ ]:
start = time.time()

et_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

et_model.fit(X_train, y_train)
y_pred_et = et_model.predict(X_test)

print("ExtraTrees F1:", f1_score(y_test, y_pred_et))
print("Time:", time.time() - start)

NameError: name 'time' is not defined

In [ ]:
import lightgbm as lgb

start = time.time()

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31
)

lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)

print("LightGBM F1:", f1_score(y_test, y_pred_lgb))
print("Time:", time.time() - start)
print(classification_report(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))

In [ ]:
import xgboost as xgb

start = time.time()

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    tree_method="hist"
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost F1:", f1_score(y_test, y_pred_xgb))
print("Time:", time.time() - start)
print(classification_report(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))

In [ ]:
def hybrid_predict(sample):
    sample_scaled = scaler.transform(sample)

    iso_pred = iso_model.predict(sample_scaled)

    if iso_pred[0] == 1:
        return "BENIGN"
    else:
        return rf_model.predict(sample_scaled)[0]